# Attention Lab 3 · Self vs Cross-Attention

> **Types of Attention — Lab 3.** Run top to bottom (Runtime → Run all). Read the plain-English notes, run the code,
> then do the **Your turn 🧪** cell. Everything is built from scratch with NumPy so nothing is hidden.

### In plain English
Same idea, two wirings.

- **Self-attention:** words in **one** sentence look at each other — like a group in a room, each listening to everyone
  else in that same room. This is how a model works out that *"it"* means *"the animal"*.
- **Cross-attention:** the text being **written** looks back at a **different** source text — like a translator writing
  French while glancing at the English original.

Understanding-only models (BERT) use self-attention. Translators use both, plus cross-attention to connect output to
input.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
VIOLET, CYAN = "#6C5CE7", "#22D3EE"
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask=None):
    s = Q @ K.T / np.sqrt(K.shape[-1])
    if mask is not None: s = np.where(mask, s, -1e9)
    W = softmax(s)
    return W @ V, W

## 1 · Self-attention: Q, K, V from the SAME sequence

In [ ]:
sent = ["the","cat","sat","because","it","was","sleepy"]
X = np.array([                    # [entity, action, pronoun, state, determiner]
 [0,0,0,0,1],                     # the
 [1,0,0,0,0],                     # cat
 [0,1,0,0,0],                     # sat
 [0,0,0,0,.3],                    # because
 [.6,0,.3,0,0],                   # it
 [0,0,0,.4,.3],                   # was
 [.35,0,0,1,0],                   # sleepy
], float) * 3.0

_, W_self = attention(X, X, X)    # <- the SAME X three times
i = sent.index("it")
print('"it" attends to:', {sent[j]: round(float(W_self[i,j]),2) for j in np.argsort(-W_self[i])[:3]})

## 2 · Cross-attention: Q from the target, K/V from the source
Here a French decoder attends to an English encoder. Notice `Q` and `K` come from *different* sequences.

In [ ]:
src = ["the","cat","drinks","the","milk"]        # English  (encoder -> Keys, Values)
tgt = ["le","chat","boit","le","lait"]           # French   (decoder -> Queries)

# toy "concept" ids: matching concepts should align
src_concept = np.array([0,1,2,0,3]); tgt_concept = np.array([0,1,2,0,3])
def onehot(ids, n=4): 
    M = np.zeros((len(ids), n)); M[np.arange(len(ids)), ids] = 1; return M

K_src = V_src = onehot(src_concept) * 3.0
Q_tgt = onehot(tgt_concept) * 3.0

out, W_cross = attention(Q_tgt, K_src, V_src)     # queries from target, keys/values from source
print("cross-attention matrix shape:", W_cross.shape, "= (target words, source words)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4.6))
a = ax[0].imshow(W_self, cmap="BuPu")
ax[0].set_xticks(range(len(sent)), sent, rotation=45, ha="right"); ax[0].set_yticks(range(len(sent)), sent)
ax[0].set_title("SELF-attention (square: one sentence)")

b = ax[1].imshow(W_cross, cmap="BuPu")
ax[1].set_xticks(range(len(src)), src, rotation=45, ha="right"); ax[1].set_yticks(range(len(tgt)), tgt)
ax[1].set_xlabel("source (English) — keys/values"); ax[1].set_ylabel("target (French) — queries")
ax[1].set_title("CROSS-attention (an alignment)")
plt.tight_layout(); plt.show()

for i, w in enumerate(tgt):
    j = W_cross[i].argmax()
    print(f"  {w:6s} aligns to  {src[j]}")

## 3 · Where each shows up
| Model | Self | Cross |
|---|---|---|
| BERT (understanding) | ✅ bidirectional | ❌ |
| GPT (generation) | ✅ causal (masked) | ❌ |
| T5 / translation | ✅ both sides | ✅ decoder → encoder |

## Your turn 🧪
1. Shuffle the French word order. Does the cross-attention alignment follow the words?
2. Make the cross-attention matrix non-square by dropping a source word — confirm the shape is `(len(tgt), len(src))`.
3. In self-attention, what does the diagonal of `W_self` mean? Why is it usually strong?

In [ ]:
# Your turn: the diagonal = how much each word attends to itself
print("self-attention diagonal:", np.round(np.diag(W_self), 2))